# 16. LaMa Baseline Report Generation

This notebook generates a standalone HTML report for the LaMa restoration baseline on the controlled 50-painting subset.

The report consolidates:

- LaMa restoration metadata,
- classical metric results,
- difference/error-map diagnostics,
- LPIPS perceptual metrics,
- CLIP and DINOv2 feature-space metrics,
- selected diagnostic cases.

The purpose is to summarize LaMa behavior before direct model comparison with OpenCV Telea.

## Report design

The report focuses on non-zero damage cases because the goal is to evaluate restoration behavior under synthetic damage.

Zero-control cases remain important sanity checks in the metric notebooks, but they are excluded from the main diagnostic report tables because they contain no damaged region.

The report uses:

- masked-region classical metrics,
- mask-bounding-box LPIPS metrics,
- mask-bounding-box CLIP/DINOv2 feature-similarity metrics,
- selected error-map figures.

This keeps the report focused on the damaged local region while preserving links to the full metric outputs.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from restoration_eval.reporting import (
    build_selected_case_sections_html,
    dataframe_to_html_table,
    html_escape,
    image_block,
    prepare_opencv_50_report_dataframe,
    select_opencv_50_diagnostic_cases,
    summarize_metric_correlations,
    summarize_report_by_category,
    summarize_report_by_mask_type,
    summarize_report_overview,
)

print("Project root:", PROJECT_ROOT)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval


In [2]:
config_path = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

paths_cfg = config["paths"]

processed_metadata_dir = PROJECT_ROOT / paths_cfg["processed_metadata_dir"]
metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
reports_dir = PROJECT_ROOT / paths_cfg["reports_dir"]

reports_dir.mkdir(parents=True, exist_ok=True)

processed_metadata_path = processed_metadata_dir / "metadata_processed_clean.csv"
lama_metadata_path = processed_metadata_dir / "metadata_restored_lama.csv"

classical_metrics_path = metrics_dir / "classical_metrics_lama_50.csv"
lpips_metrics_path = metrics_dir / "lpips_metrics_lama_50.csv"
feature_metrics_path = metrics_dir / "feature_similarity_lama_50.csv"
error_map_manifest_path = metrics_dir / "error_map_manifest_all_lama_50.csv"

report_output_path = reports_dir / "lama_baseline_report_50.html"
selected_cases_output_path = metrics_dir / "lama_report_selected_cases_50.csv"
report_dataframe_output_path = metrics_dir / "lama_report_dataframe_50.csv"

model_name = "lama"

print("Processed metadata:", processed_metadata_path)
print("LaMa metadata:", lama_metadata_path)
print("Classical metrics:", classical_metrics_path)
print("LPIPS metrics:", lpips_metrics_path)
print("Feature metrics:", feature_metrics_path)
print("Error-map manifest:", error_map_manifest_path)
print("Report output:", report_output_path)

Processed metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_processed_clean.csv
LaMa metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restored_lama.csv
Classical metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\classical_metrics_lama_50.csv
LPIPS metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lpips_metrics_lama_50.csv
Feature metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\feature_similarity_lama_50.csv
Error-map manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\error_map_manifest_all_lama_50.csv
Report output: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\lama_baseline_report_50.html


In [3]:
required_input_paths = [
    processed_metadata_path,
    lama_metadata_path,
    classical_metrics_path,
    lpips_metrics_path,
    feature_metrics_path,
    error_map_manifest_path,
]

for input_path in required_input_paths:
    if not input_path.exists():
        raise FileNotFoundError(f"Missing required input: {input_path}")

processed_metadata_df = pd.read_csv(processed_metadata_path)
lama_restored_df = pd.read_csv(lama_metadata_path)
classical_metrics_df = pd.read_csv(classical_metrics_path)
lpips_metrics_df = pd.read_csv(lpips_metrics_path)
feature_metrics_df = pd.read_csv(feature_metrics_path)
error_map_manifest_df = pd.read_csv(error_map_manifest_path)

print("Processed metadata shape:", processed_metadata_df.shape)
print("LaMa restored metadata shape:", lama_restored_df.shape)
print("Classical metrics shape:", classical_metrics_df.shape)
print("LPIPS metrics shape:", lpips_metrics_df.shape)
print("Feature metrics shape:", feature_metrics_df.shape)
print("Error-map manifest shape:", error_map_manifest_df.shape)

print("\nLaMa restored model names:")
display(lama_restored_df["model_name"].value_counts(dropna=False))

print("\nClassical metric regions:")
display(classical_metrics_df["evaluation_region"].value_counts())

print("\nLPIPS metric regions:")
display(lpips_metrics_df["evaluation_region"].value_counts())

print("\nFeature metric regions:")
display(feature_metrics_df["evaluation_region"].value_counts())

print("\nError-map manifest status:")
display(error_map_manifest_df["status"].value_counts(dropna=False))

Processed metadata shape: (50, 43)
LaMa restored metadata shape: (250, 32)
Classical metrics shape: (900, 30)
LPIPS metrics shape: (700, 24)
Feature metrics shape: (700, 28)
Error-map manifest shape: (250, 29)

LaMa restored model names:


model_name
lama    250
Name: count, dtype: int64


Classical metric regions:


evaluation_region
full_image        250
content_region    250
masked_region     200
mask_bbox_crop    200
Name: count, dtype: int64


LPIPS metric regions:


evaluation_region
full_image        250
content_region    250
mask_bbox_crop    200
Name: count, dtype: int64


Feature metric regions:


evaluation_region
full_image        250
content_region    250
mask_bbox_crop    200
Name: count, dtype: int64


Error-map manifest status:


status
ok    250
Name: count, dtype: int64

In [4]:
expected_shapes = {
    "processed_metadata": 50,
    "lama_restored": 250,
    "classical_metrics": 900,
    "lpips_metrics": 700,
    "feature_metrics": 700,
    "error_map_manifest": 250,
}

actual_shapes = {
    "processed_metadata": len(processed_metadata_df),
    "lama_restored": len(lama_restored_df),
    "classical_metrics": len(classical_metrics_df),
    "lpips_metrics": len(lpips_metrics_df),
    "feature_metrics": len(feature_metrics_df),
    "error_map_manifest": len(error_map_manifest_df),
}

print("Expected row counts:", expected_shapes)
print("Actual row counts:", actual_shapes)

for name, expected_count in expected_shapes.items():
    actual_count = actual_shapes[name]
    if actual_count != expected_count:
        raise ValueError(
            f"{name}: expected {expected_count} rows, found {actual_count}."
        )

if set(lama_restored_df["model_name"].dropna().unique()) != {model_name}:
    raise ValueError(
        f"Unexpected restored model names: {lama_restored_df['model_name'].unique()}"
    )

for dataframe_name, dataframe in [
    ("lama_restored_df", lama_restored_df),
    ("classical_metrics_df", classical_metrics_df),
    ("lpips_metrics_df", lpips_metrics_df),
    ("feature_metrics_df", feature_metrics_df),
    ("error_map_manifest_df", error_map_manifest_df),
]:
    if "status" in dataframe.columns:
        non_ok_rows = int((dataframe["status"] != "ok").sum())
        print(f"{dataframe_name} non-ok rows:", non_ok_rows)

        if non_ok_rows != 0:
            display(dataframe[dataframe["status"] != "ok"].head(20))
            raise ValueError(f"{dataframe_name} contains non-ok rows.")

expected_classical_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "masked_region": 200,
    "mask_bbox_crop": 200,
}

actual_classical_region_counts = classical_metrics_df["evaluation_region"].value_counts().to_dict()

print("\nExpected classical region counts:", expected_classical_region_counts)
print("Actual classical region counts:", actual_classical_region_counts)

for region, expected_count in expected_classical_region_counts.items():
    actual_count = actual_classical_region_counts.get(region, 0)
    if actual_count != expected_count:
        raise ValueError(
            f"Classical region {region!r}: expected {expected_count}, found {actual_count}."
        )

expected_spatial_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "mask_bbox_crop": 200,
}

for dataframe_name, dataframe in [
    ("lpips_metrics_df", lpips_metrics_df),
    ("feature_metrics_df", feature_metrics_df),
]:
    actual_region_counts = dataframe["evaluation_region"].value_counts().to_dict()

    print(f"\nExpected {dataframe_name} region counts:", expected_spatial_region_counts)
    print(f"Actual {dataframe_name} region counts:", actual_region_counts)

    for region, expected_count in expected_spatial_region_counts.items():
        actual_count = actual_region_counts.get(region, 0)
        if actual_count != expected_count:
            raise ValueError(
                f"{dataframe_name} region {region!r}: "
                f"expected {expected_count}, found {actual_count}."
            )

print("\nLaMa report input gates passed.")

Expected row counts: {'processed_metadata': 50, 'lama_restored': 250, 'classical_metrics': 900, 'lpips_metrics': 700, 'feature_metrics': 700, 'error_map_manifest': 250}
Actual row counts: {'processed_metadata': 50, 'lama_restored': 250, 'classical_metrics': 900, 'lpips_metrics': 700, 'feature_metrics': 700, 'error_map_manifest': 250}
lama_restored_df non-ok rows: 0
classical_metrics_df non-ok rows: 0
lpips_metrics_df non-ok rows: 0
feature_metrics_df non-ok rows: 0
error_map_manifest_df non-ok rows: 0

Expected classical region counts: {'full_image': 250, 'content_region': 250, 'masked_region': 200, 'mask_bbox_crop': 200}
Actual classical region counts: {'full_image': 250, 'content_region': 250, 'masked_region': 200, 'mask_bbox_crop': 200}

Expected lpips_metrics_df region counts: {'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}
Actual lpips_metrics_df region counts: {'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}

Expected feature_metrics_df region c

In [5]:
report_df = prepare_opencv_50_report_dataframe(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=lama_restored_df,
    classical_metrics_df=classical_metrics_df,
    lpips_metrics_df=lpips_metrics_df,
    feature_metrics_df=feature_metrics_df,
    error_map_manifest_df=error_map_manifest_df,
    project_root=PROJECT_ROOT,
    include_zero_control=False,
)

print("LaMa report dataframe shape:", report_df.shape)

display(report_df.head())

print("\nReport mask type counts:")
display(report_df["mask_type"].value_counts().sort_index())

print("\nReport category counts:")
display(report_df["category"].value_counts().sort_index())

print("\nReport model names:")
display(report_df["model_name"].value_counts(dropna=False))

required_report_columns = [
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
    "model_name",
    "mse_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "error_map_figure_path",
]

missing_report_columns = [
    column for column in required_report_columns
    if column not in report_df.columns
]

if missing_report_columns:
    raise ValueError(
        f"Report dataframe missing required columns: {missing_report_columns}"
    )

if len(report_df) != 200:
    raise ValueError(
        f"Expected 200 non-zero report cases, found {len(report_df)}."
    )

if set(report_df["model_name"].dropna().unique()) != {model_name}:
    raise ValueError(
        f"Unexpected report model names: {report_df['model_name'].unique()}"
    )

if report_df["category"].isna().any():
    raise ValueError("Report dataframe contains missing category labels.")

if report_df["error_map_figure_path"].isna().any():
    raise ValueError("Report dataframe contains missing error-map figure path values.")

missing_error_map_paths = [
    path
    for path in report_df["error_map_figure_path"]
    if str(path).strip() == "" or not Path(path).exists()
]

if missing_error_map_paths:
    raise FileNotFoundError(
        f"Missing error-map figures referenced by report dataframe: "
        f"{missing_error_map_paths[:10]}"
    )

print("\nLaMa report dataframe passed validation.")

LaMa report dataframe shape: (200, 61)


,case_id,painting_id,mask_id,mask_type,clean_filename,clean_path,mask_filename,mask_path,damaged_filename,damaged_path,...,damaged_lpips,restored_lpips,lpips_improvement,clip_damaged_similarity,clip_restored_similarity,clip_similarity_improvement,dinov2_damaged_similarity,dinov2_restored_similarity,dinov2_similarity_improvement,error_map_figure_path
0,p001_loss_large,p001,p001_loss_large,loss_large,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_large_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_large_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,0.592107,0.147469,0.444638,0.810900,0.977687,0.166787,0.706465,0.960092,0.253627,D:\Masters\FH\Thesis\painting-restoration-eval...
1,p001_loss_small,p001,p001_loss_small,loss_small,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_small_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_small_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,0.341009,0.002937,0.338072,0.910694,0.999803,0.089109,0.948942,0.999632,0.050690,D:\Masters\FH\Thesis\painting-restoration-eval...
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_mixed_damage_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_mixed_damage_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,0.416328,0.024558,0.391770,0.895807,0.998196,0.102389,0.795198,0.992431,0.197233,D:\Masters\FH\Thesis\painting-restoration-eval...
3,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_scratch_thin_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_scratch_thin_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,0.425347,0.005050,0.420297,0.857966,0.999694,0.141728,0.924942,0.999538,0.074596,D:\Masters\FH\Thesis\painting-restoration-eval...
4,p002_loss_large,p002,p002_loss_large,loss_large,p002_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p002_loss_large_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p002_loss_large_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,0.589203,0.140967,0.448236,0.724333,0.969885,0.245552,0.762878,0.935186,0.172309,D:\Masters\FH\Thesis\painting-restoration-eval...



Report mask type counts:


mask_type
loss_large      50
loss_small      50
mixed_damage    50
scratch_thin    50
Name: count, dtype: int64


Report category counts:


category
abstraction_surrealism     40
architecture_structured    40
high_texture_brushwork     40
landscape_natural          40
portrait_figure            40
Name: count, dtype: int64


Report model names:


model_name
lama    200
Name: count, dtype: int64


LaMa report dataframe passed validation.


In [6]:
overview_df = summarize_report_overview(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=lama_restored_df,
    report_df=report_df,
)

summary_by_mask_df = summarize_report_by_mask_type(report_df)
summary_by_category_df = summarize_report_by_category(report_df)
correlation_df = summarize_metric_correlations(report_df)

selected_cases_df = select_opencv_50_diagnostic_cases(
    report_df,
    n_per_group=3,
    include_category_examples=True,
)

report_df.to_csv(report_dataframe_output_path, index=False)
selected_cases_df.to_csv(selected_cases_output_path, index=False)

print("Overview:")
display(overview_df)

print("\nSummary by mask type:")
display(summary_by_mask_df)

print("\nSummary by category:")
display(summary_by_category_df)

print("\nMetric correlation matrix:")
display(correlation_df)

print("\nSelected diagnostic cases:", len(selected_cases_df))
display(
    selected_cases_df[
        [
            "case_id",
            "painting_id",
            "category",
            "title",
            "mask_type",
            "selection_reason",
            "mse_improvement",
            "lpips_improvement",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
            "error_map_figure_path",
        ]
    ]
)

print("\nSaved report dataframe:")
print(report_dataframe_output_path)

print("\nSaved selected cases:")
print(selected_cases_output_path)

Overview:


,item,value
0,Paintings,50
1,Painting categories,5
2,Restoration cases generated,250
3,Non-zero report cases,200
4,Mask types,5
5,Baseline model,lama



Summary by mask type:


,mask_type,cases,mean_damage_area_content_pct,mean_mse_improvement,mean_restored_mse,mean_lpips_improvement,mean_restored_lpips,mean_clip_improvement,clip_improvement_rate,mean_dinov2_improvement,dinov2_improvement_rate
0,loss_large,50,12.82220,24555.37581,1025.34706,0.30796,0.17733,0.15819,1.0,0.13550,0.98
1,loss_small,50,4.43850,26147.75371,372.12671,0.18263,0.01011,0.10370,1.0,0.07885,1.00
2,mixed_damage,50,10.13509,26669.08976,568.50417,0.29051,0.02337,0.14980,1.0,0.23192,1.00
3,scratch_thin,50,2.19846,28387.47347,257.25934,0.25820,0.00201,0.10123,1.0,0.14685,1.00



Summary by category:


...


Metric correlation matrix:


,mse_improvement,mae_improvement,psnr_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement
mse_improvement,1.0000,0.9817,0.5966,0.4450,0.1376,0.0789
mae_improvement,0.9817,1.0000,0.6498,0.4566,0.1549,0.0961
psnr_improvement,0.5966,0.6498,1.0000,0.3169,0.0022,0.0249
lpips_improvement,0.4450,0.4566,0.3169,1.0000,0.6137,0.5237
clip_similarity_improvement,0.1376,0.1549,0.0022,0.6137,1.0000,0.3942
dinov2_similarity_improvement,0.0789,0.0961,0.0249,0.5237,0.3942,1.0000



Selected diagnostic cases: 21


,case_id,painting_id,category,title,mask_type,selection_reason,mse_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement,error_map_figure_path
0,p002_loss_large,p002,portrait_figure,Madame X (Madame Pierre Gautreau),loss_large,Strongest masked-region MSE improvement,51135.079025,0.448236,0.245552,0.172309,D:\Masters\FH\Thesis\painting-restoration-eval...
1,p006_loss_large,p006,portrait_figure,Boy with a Sword,loss_large,Strong category example by MSE improvement; St...,52039.680618,0.439890,0.059199,0.102409,D:\Masters\FH\Thesis\painting-restoration-eval...
2,p006_mixed_damage,p006,portrait_figure,Boy with a Sword,mixed_damage,Strongest masked-region MSE improvement,51938.140059,0.424019,0.163723,0.219838,D:\Masters\FH\Thesis\painting-restoration-eval...
3,p006_scratch_thin,p006,portrait_figure,Boy with a Sword,scratch_thin,Strongest mask-bbox LPIPS improvement,51037.976669,0.465335,0.128275,0.188642,D:\Masters\FH\Thesis\painting-restoration-eval...
4,p010_mixed_damage,p010,portrait_figure,Joan of Arc,mixed_damage,Strongest mask-bbox LPIPS improvement,40033.681091,0.480820,0.231196,0.518256,D:\Masters\FH\Thesis\painting-restoration-eval...
5,p011_loss_large,p011,landscape_natural,View of Haarlem and the Haarlemmer Meer,loss_large,Strongest mask-bbox LPIPS improvement,19176.598892,0.533650,0.182890,0.472010,D:\Masters\FH\Thesis\painting-restoration-eval...
6,p013_loss_large,p013,landscape_natural,Landscape with a Village in the Distance,loss_large,Strong category example by MSE improvement,45390.057556,0.418887,0.222950,0.275566,D:\Masters\FH\Thesis\painting-restoration-eval...
7,p014_mixed_damage,p014,landscape_natural,Landscape on a River,mixed_damage,Strongest DINOv2 feature improvement,24338.505005,0.416432,0.189679,0.657182,D:\Masters\FH\Thesis\painting-restoration-eval...
8,p022_loss_small,p022,architecture_structured,"Interior of a Protestant, Gothic Church, with ...",loss_small,Strong category example by MSE improvement,36992.959351,0.249487,0.220946,0.057840,D:\Masters\FH\Thesis\painting-restoration-eval...
9,p032_mixed_damage,p032,abstraction_surrealism,Composition (No. 1) Gray-Red,mixed_damage,Weakest masked-region MSE improvement,2734.126339,0.209575,0.063350,0.144347,D:\Masters\FH\Thesis\painting-restoration-eval...



Saved report dataframe:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lama_report_dataframe_50.csv

Saved selected cases:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lama_report_selected_cases_50.csv


In [7]:
def interpret_lama_case(row: pd.Series) -> str:
    """Generate a short LaMa-specific interpretation for one selected case."""
    mask_type = row.get("mask_type", "")
    mse_improvement = row.get("mse_improvement", pd.NA)
    lpips_improvement = row.get("lpips_improvement", pd.NA)
    clip_improvement = row.get("clip_similarity_improvement", pd.NA)
    dinov2_improvement = row.get("dinov2_similarity_improvement", pd.NA)

    comments = []

    if mask_type == "scratch_thin":
        comments.append(
            "This thin-scratch case tests whether LaMa can remove narrow artificial damage while preserving local image structure."
        )
    elif mask_type == "loss_small":
        comments.append(
            "This small-loss case tests whether LaMa can reconstruct a compact missing region using surrounding visual context."
        )
    elif mask_type == "loss_large":
        comments.append(
            "This large-loss case is more demanding because the missing region may require broader contextual or semantic reconstruction."
        )
    elif mask_type == "mixed_damage":
        comments.append(
            "This mixed-damage case combines several damage patterns and tests whether LaMa remains stable under compound degradation."
        )

    if pd.notna(mse_improvement):
        if mse_improvement > 10_000:
            comments.append(
                "The masked-region MSE improvement is large, indicating that the restored pixels are much closer to the clean reference than the white-filled damaged input."
            )
        elif mse_improvement > 0:
            comments.append(
                "The masked-region MSE improvement is positive, indicating numerical improvement over the damaged input."
            )
        else:
            comments.append(
                "The masked-region MSE improvement is non-positive, indicating a failure under this pixel-level metric."
            )

    if pd.notna(lpips_improvement):
        if lpips_improvement > 0:
            comments.append(
                "LPIPS improves, suggesting that the restored local crop is perceptually closer to the clean reference."
            )
        else:
            comments.append(
                "LPIPS does not improve, suggesting that numerical restoration did not translate into local perceptual improvement."
            )

    if pd.notna(clip_improvement):
        if clip_improvement > 0:
            comments.append(
                "CLIP feature similarity improves, suggesting better high-level visual alignment with the clean reference."
            )
        else:
            comments.append(
                "CLIP feature similarity decreases, suggesting that the restored output may not improve broad feature-space alignment."
            )

    if pd.notna(dinov2_improvement):
        if dinov2_improvement > 0:
            comments.append(
                "DINOv2 feature similarity improves, suggesting better alignment with local visual structure in a self-supervised feature space."
            )
        else:
            comments.append(
                "DINOv2 feature similarity decreases, indicating possible local structural inconsistency despite visible damage removal."
            )

    return " ".join(comments)


def build_lama_key_findings_html(
    *,
    summary_by_mask: pd.DataFrame,
    summary_by_category: pd.DataFrame,
    correlation_df: pd.DataFrame,
) -> str:
    """Build LaMa-specific textual findings for the report."""
    strongest_mask_by_mse = summary_by_mask.sort_values(
        "mean_mse_improvement",
        ascending=False,
    ).iloc[0]

    weakest_mask_by_dino = summary_by_mask.sort_values(
        "mean_dinov2_improvement",
        ascending=True,
    ).iloc[0]

    weakest_category_by_dino = summary_by_category.sort_values(
        "mean_dinov2_improvement",
        ascending=True,
    ).iloc[0]

    strongest_category_by_lpips = summary_by_category.sort_values(
        "mean_lpips_improvement",
        ascending=False,
    ).iloc[0]

    corr_html = dataframe_to_html_table(
        correlation_df.reset_index().rename(columns={"index": "metric"}),
        float_decimals=4,
    )

    return f"""
    <h2>Key findings</h2>

    <p>
        LaMa generally reduces the visible white-mask damage under controlled synthetic
        degradation. The strongest average masked-region MSE improvement was observed for
        <b>{html_escape(strongest_mask_by_mse['mask_type'])}</b>.
    </p>

    <p>
        However, pixel-level improvement is not sufficient evidence of faithful restoration.
        The weakest average DINOv2 feature-space behavior was observed for
        <b>{html_escape(weakest_mask_by_dino['mask_type'])}</b>, with mean DINOv2 improvement of
        <b>{float(weakest_mask_by_dino['mean_dinov2_improvement']):.5f}</b>.
        This indicates that learned inpainting can reduce local pixel error while still producing
        outputs that do not always improve structural feature similarity.
    </p>

    <p>
        Category-level behavior also varies. The weakest average DINOv2 improvement was observed for
        <b>{html_escape(weakest_category_by_dino['category'])}</b>, while the strongest average LPIPS
        improvement was observed for <b>{html_escape(strongest_category_by_lpips['category'])}</b>.
        This supports the decision to evaluate across painting categories instead of treating the
        dataset as one homogeneous image set.
    </p>

    <p>
        The metric correlations show whether classical, perceptual, and feature-space signals agree
        or diverge. Such disagreement is methodologically important: it shows that restoration
        behavior cannot be summarized reliably by a single scalar metric family.
    </p>

    <h3>Main metric correlation matrix</h3>
    {corr_html}
    """

In [8]:
def build_lama_selected_case_sections_html(
    selected_cases_df: pd.DataFrame,
    *,
    project_root: Path,
    image_mode: str = "linked",
    image_width: int = 980,
) -> str:
    """Build selected LaMa diagnostic case sections using error-map figures."""
    if selected_cases_df.empty:
        return "<p>No selected diagnostic cases were provided.</p>"

    sections = []

    for _, row in selected_cases_df.iterrows():
        figure_path_value = row.get("error_map_figure_path", "")

        if pd.notna(figure_path_value) and str(figure_path_value).strip() != "":
            figure_html = image_block(
                Path(str(figure_path_value)),
                caption="Diagnostic error-map figure",
                project_root=project_root,
                width=image_width,
                mode=image_mode,
            )
        else:
            figure_html = """
            <p class="missing">
                No error-map figure path available for this case.
            </p>
            """

        metric_table = pd.DataFrame(
            [
                {
                    "case_id": row.get("case_id", ""),
                    "category": row.get("category", ""),
                    "mask_type": row.get("mask_type", ""),
                    "mse_improvement": row.get("mse_improvement", pd.NA),
                    "lpips_improvement": row.get("lpips_improvement", pd.NA),
                    "clip_improvement": row.get("clip_similarity_improvement", pd.NA),
                    "dinov2_improvement": row.get("dinov2_similarity_improvement", pd.NA),
                }
            ]
        )

        sections.append(
            f"""
            <section class="case-section">
                <h3>{html_escape(row.get("case_id", ""))}: {html_escape(row.get("title", ""))}</h3>

                <p>
                    <b>Selection reason:</b> {html_escape(row.get("selection_reason", ""))}<br>
                    <b>Painting ID:</b> {html_escape(row.get("painting_id", ""))}<br>
                    <b>Category:</b> {html_escape(row.get("category", ""))}<br>
                    <b>Mask type:</b> {html_escape(row.get("mask_type", ""))}<br>
                    <b>Artist:</b> {html_escape(row.get("artist", ""))}<br>
                    <b>Date:</b> {html_escape(row.get("date", ""))}
                </p>

                {dataframe_to_html_table(metric_table, float_decimals=5)}

                <p><b>Interpretation:</b> {html_escape(interpret_lama_case(row))}</p>

                {figure_html}
            </section>
            """
        )

    return "\n".join(sections)

In [9]:
def build_lama_50_report_html(
    *,
    overview_df: pd.DataFrame,
    summary_by_mask: pd.DataFrame,
    summary_by_category: pd.DataFrame,
    correlation_df: pd.DataFrame,
    selected_cases_df: pd.DataFrame,
    project_root: Path,
    image_mode: str = "linked",
    title: str = "LaMa 50-Painting Baseline Report",
) -> str:
    """Build the full LaMa 50-painting HTML report."""
    overview_html = dataframe_to_html_table(overview_df, float_decimals=4)
    mask_summary_html = dataframe_to_html_table(summary_by_mask, float_decimals=5)
    category_summary_html = dataframe_to_html_table(summary_by_category, float_decimals=5)

    key_findings_html = build_lama_key_findings_html(
        summary_by_mask=summary_by_mask,
        summary_by_category=summary_by_category,
        correlation_df=correlation_df,
    )

    selected_cases_html = build_lama_selected_case_sections_html(
        selected_cases_df,
        project_root=project_root,
        image_mode=image_mode,
    )

    return f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>{html_escape(title)}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: #f7f7f7;
                color: #222;
                line-height: 1.55;
            }}

            h1 {{
                color: #111;
                border-bottom: 2px solid #333;
                padding-bottom: 10px;
            }}

            h2 {{
                margin-top: 38px;
                color: #222;
            }}

            h3 {{
                margin-top: 26px;
                color: #333;
            }}

            .summary,
            .case-section {{
                background: white;
                padding: 22px;
                border-radius: 8px;
                margin-bottom: 30px;
                border: 1px solid #ddd;
            }}

            .case-section {{
                page-break-inside: avoid;
            }}

            .image-block {{
                text-align: center;
                font-size: 12px;
                margin-top: 15px;
            }}

            .image-block img {{
                border: 1px solid #ccc;
                background: #eee;
                max-width: 100%;
                height: auto;
            }}

            .caption {{
                margin-top: 6px;
                color: #555;
            }}

            .missing {{
                color: #9a3412;
                font-style: italic;
            }}

            table,
            .summary-table {{
                border-collapse: collapse;
                margin-top: 15px;
                margin-bottom: 18px;
                width: 100%;
                font-size: 13px;
                background: white;
            }}

            th,
            td,
            .summary-table th,
            .summary-table td {{
                border: 1px solid #ccc;
                padding: 7px;
                text-align: center;
                vertical-align: middle;
            }}

            th,
            .summary-table th {{
                background-color: #eee;
                font-weight: bold;
            }}

            .note {{
                background: #fff7ed;
                border: 1px solid #fed7aa;
                padding: 12px;
                border-radius: 6px;
            }}
        </style>
    </head>

    <body>
        <h1>{html_escape(title)}</h1>

        <div class="summary">
            <h2>Experiment overview</h2>
            <p>
                This report consolidates the LaMa baseline evaluation for the controlled
                50-painting subset. It summarizes restoration behavior across synthetic
                damage types, painting categories, and multiple metric families.
            </p>

            {overview_html}

            <p class="note">
                LaMa is used here as a pretrained learned inpainting baseline. The goal is
                not to claim historically faithful restoration, but to evaluate how a learned
                inpainting model behaves under the proposed controlled evaluation framework.
            </p>
        </div>

        <div class="summary">
            <h2>Summary by mask type</h2>
            {mask_summary_html}

            <h2>Summary by painting category</h2>
            {category_summary_html}

            {key_findings_html}
        </div>

        <div class="summary">
            <h2>Selected diagnostic cases</h2>
            <p>
                The following cases were selected from strongest and weakest metric outcomes,
                feature-space behavior, and category examples. They support qualitative inspection
                but do not replace the full metric tables.
            </p>

            {selected_cases_html}
        </div>

        <div class="summary">
            <h2>Baseline conclusion</h2>
            <p>
                The LaMa baseline provides a learned inpainting comparison point after the
                deterministic OpenCV Telea baseline. Its outputs can reduce synthetic damage
                under classical and perceptual metrics, but feature-space and spatial diagnostics
                remain necessary to identify local failures, smoothing, structural inconsistency,
                or metric disagreement.
            </p>

            <p>
                The disagreement between MSE, LPIPS, CLIP, DINOv2, and visual error maps is treated
                as a central result of the evaluation framework rather than a reporting problem.
                It shows why restoration outputs should be assessed through complementary metric
                families and visual diagnostics.
            </p>

            <p>
                This report prepares the LaMa baseline for direct comparison with OpenCV Telea
                and for later extension to diffusion-based inpainting models.
            </p>
        </div>
    </body>
    </html>
    """

In [12]:
html_report = build_lama_50_report_html(
    overview_df=overview_df,
    summary_by_mask=summary_by_mask_df,
    summary_by_category=summary_by_category_df,
    correlation_df=correlation_df,
    selected_cases_df=selected_cases_df,
    project_root=PROJECT_ROOT,
    image_mode="embedded",
    title="LaMa 50-Painting Baseline Report",
)

report_output_path.parent.mkdir(parents=True, exist_ok=True)
report_output_path.write_text(html_report, encoding="utf-8")

print("Saved LaMa HTML report:")
print(report_output_path)

print("Report size in characters:", len(html_report))

Saved LaMa HTML report:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\lama_baseline_report_50.html
Report size in characters: 37032633


In [13]:
expected_report_files = [
    report_output_path,
    selected_cases_output_path,
    report_dataframe_output_path,
]

for output_file in expected_report_files:
    if not output_file.exists():
        raise FileNotFoundError(f"Missing expected report output: {output_file}")

saved_report_df = pd.read_csv(report_dataframe_output_path)
saved_selected_cases_df = pd.read_csv(selected_cases_output_path)

if len(saved_report_df) != 200:
    raise ValueError(
        f"Expected 200 report dataframe rows, found {len(saved_report_df)}."
    )

if len(saved_selected_cases_df) == 0:
    raise ValueError("Selected diagnostic cases file is empty.")

if saved_report_df["category"].isna().any():
    raise ValueError("Saved report dataframe contains missing category labels.")

if saved_report_df["model_name"].nunique() != 1 or saved_report_df["model_name"].iloc[0] != model_name:
    raise ValueError(
        f"Unexpected model names in saved report dataframe: "
        f"{saved_report_df['model_name'].unique()}"
    )

expected_report_mask_counts = {
    "loss_large": 50,
    "loss_small": 50,
    "mixed_damage": 50,
    "scratch_thin": 50,
}

actual_report_mask_counts = saved_report_df["mask_type"].value_counts().to_dict()

print("Expected report mask counts:", expected_report_mask_counts)
print("Actual report mask counts:", actual_report_mask_counts)

for mask_type, expected_count in expected_report_mask_counts.items():
    actual_count = actual_report_mask_counts.get(mask_type, 0)
    if actual_count != expected_count:
        raise ValueError(
            f"Report mask type {mask_type!r}: expected {expected_count}, found {actual_count}."
        )

expected_report_category_counts = {
    "portrait_figure": 40,
    "landscape_natural": 40,
    "architecture_structured": 40,
    "abstraction_surrealism": 40,
    "high_texture_brushwork": 40,
}

actual_report_category_counts = saved_report_df["category"].value_counts().to_dict()

print("\nExpected report category counts:", expected_report_category_counts)
print("Actual report category counts:", actual_report_category_counts)

for category, expected_count in expected_report_category_counts.items():
    actual_count = actual_report_category_counts.get(category, 0)
    if actual_count != expected_count:
        raise ValueError(
            f"Report category {category!r}: expected {expected_count}, found {actual_count}."
        )

required_metric_columns = [
    "mse_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
]

for metric_column in required_metric_columns:
    if saved_report_df[metric_column].isna().any():
        raise ValueError(
            f"Saved report dataframe contains missing values in {metric_column!r}."
        )

missing_selected_error_maps = [
    path
    for path in saved_selected_cases_df["error_map_figure_path"]
    if str(path).strip() == "" or not Path(path).exists()
]

if missing_selected_error_maps:
    raise FileNotFoundError(
        f"Selected cases reference missing error-map figures: "
        f"{missing_selected_error_maps[:10]}"
    )

html_text = report_output_path.read_text(encoding="utf-8")

required_html_phrases = [
    "LaMa 50-Painting Baseline Report",
    "Experiment overview",
    "Summary by mask type",
    "Summary by painting category",
    "Selected diagnostic cases",
    "Baseline conclusion",
]

missing_html_phrases = [
    phrase for phrase in required_html_phrases
    if phrase not in html_text
]

if missing_html_phrases:
    raise ValueError(
        f"Generated HTML report missing expected phrases: {missing_html_phrases}"
    )

for forbidden_phrase in [
    "OpenCV Telea 50-Painting Baseline Report",
    "OpenCV Telea is used here",
]:
    if forbidden_phrase in html_text:
        raise ValueError(
            f"Generated LaMa report contains OpenCV-specific phrase: {forbidden_phrase!r}"
        )

print("\nSaved report dataframe rows:", len(saved_report_df))
print("Saved selected diagnostic cases:", len(saved_selected_cases_df))
print("HTML report size:", report_output_path.stat().st_size, "bytes")
print("Final LaMa report output gates passed.")

Expected report mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}
Actual report mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}

Expected report category counts: {'portrait_figure': 40, 'landscape_natural': 40, 'architecture_structured': 40, 'abstraction_surrealism': 40, 'high_texture_brushwork': 40}
Actual report category counts: {'portrait_figure': 40, 'landscape_natural': 40, 'architecture_structured': 40, 'abstraction_surrealism': 40, 'high_texture_brushwork': 40}

Saved report dataframe rows: 200
Saved selected diagnostic cases: 21
HTML report size: 37034079 bytes
Final LaMa report output gates passed.


## Notebook 16 summary

This notebook generated the standalone LaMa baseline report for the controlled 50-painting subset.

The report consolidates:

- LaMa restoration metadata,
- masked-region classical metrics,
- mask-bounding-box LPIPS metrics,
- mask-bounding-box CLIP and DINOv2 feature-space metrics,
- selected diagnostic error-map figures.

Main outputs:

- `outputs/reports/lama_baseline_report_50.html`
- `outputs/metrics/lama_report_dataframe_50.csv`
- `outputs/metrics/lama_report_selected_cases_50.csv`

The report focuses on the 200 non-zero damage cases. Zero-control cases remain part of the metric validation workflow but are excluded from the main report dataframe because they contain no damaged region.

This report completes the standalone LaMa baseline evaluation and prepares the project for direct OpenCV Telea versus LaMa comparison.